# W3 数据清洗与预处理
本 Notebook 从原始数据重新执行清洗与训练内预处理，核验 C01–C10。主表保留未知累计费用；模型矩阵的插补不是补出真实账单。

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'src/data/clean.py').is_file() and (p / 'configs/w3_cleaning.json').is_file())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.data.clean import read_raw, TelcoCleaner
from src.data.preprocess import split_xy, build_preprocessor
config = json.loads('{"version": "w3-telco-cleaning-v1.0.0", "seed": 42, "test_size": 0.2, "input_path": "data/raw/telco_customer_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv", "output_dir": "data/processed/w3", "model_dir": "models/w3", "expected_sha256": "88be4b93fbe0cc83421af1c503794c97c342eca914c1576db7c276e61d61358a", "raw_columns": ["customerID", "gender", "SeniorCitizen", "Partner", "Dependents", "tenure", "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod", "MonthlyCharges", "TotalCharges", "Churn"], "derived_columns": ["churn_label", "total_charges_missing", "tenure_zero", "internet_applicable"], "categories": {"gender": ["Female", "Male"], "SeniorCitizen": [0, 1], "Partner": ["No", "Yes"], "Dependents": ["No", "Yes"], "PhoneService": ["No", "Yes"], "MultipleLines": ["No", "Yes", "No phone service"], "InternetService": ["DSL", "Fiber optic", "No"], "OnlineSecurity": ["No", "Yes", "No internet service"], "OnlineBackup": ["No", "Yes", "No internet service"], "DeviceProtection": ["No", "Yes", "No internet service"], "TechSupport": ["No", "Yes", "No internet service"], "StreamingTV": ["No", "Yes", "No internet service"], "StreamingMovies": ["No", "Yes", "No internet service"], "Contract": ["Month-to-month", "One year", "Two year"], "PaperlessBilling": ["No", "Yes"], "PaymentMethod": ["Bank transfer (automatic)", "Credit card (automatic)", "Electronic check", "Mailed check"], "Churn": ["No", "Yes"]}, "numeric_rules": {"SeniorCitizen": {"integer": true, "minimum": 0, "minimum_inclusive": true, "maximum": 1}, "tenure": {"integer": true, "minimum": 0, "minimum_inclusive": true, "maximum": null}, "MonthlyCharges": {"integer": false, "minimum": 0, "minimum_inclusive": false, "maximum": null}, "TotalCharges": {"integer": false, "minimum": 0, "minimum_inclusive": true, "maximum": null}}, "allowed_nulls": ["TotalCharges"], "label_mapping": {"No": 0, "Yes": 1}, "internet_service_columns": ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"], "structural_categories": {"no_internet": "No", "internet_not_applicable": "No internet service", "no_phone": "No", "phone_not_applicable": "No phone service"}, "features": {"numeric": ["tenure", "MonthlyCharges", "TotalCharges"], "categorical": ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod"], "binary": ["total_charges_missing", "tenure_zero", "internet_applicable"]}, "preprocessing": {"numeric_imputation": "training_median", "all_missing_training_column_fallback": 0.0, "keep_empty_features": true, "numeric_scaling": "training_standard_scaler", "unknown_category_policy": "error"}, "semantics": {"source": "Telco public teaching proxy, not Azure internal customer data", "observation": "static snapshot; no verifiable observation or event dates", "missing_total_charges": "unknown; retained as NA with a flag in the cleaned table", "all_missing_training_column": "SimpleImputer keeps an entirely missing training column and uses computational zero; this is not a known monetary amount", "split": "stratified random interface demonstration, not a final temporal modeling split", "outlier_policy": "diagnose only; no deletion, winsorization, or sample-maximum business limits", "feature_scope": "preprocessing interface demonstration; not W4 final feature selection"}}')
source = ROOT / 'data/raw/telco_customer_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'
raw = read_raw(source)
cleaner = TelcoCleaner(config)
cleaned = cleaner.fit_transform(raw)
print('原始与清洗后:', raw.shape, cleaned.shape)

原始与清洗后: (7043, 21) (7043, 25)


## 保留所有客户与标签
空白累计费用转为 NA，并保留缺失/零年限标记；无互联网类别仍然存在。下面仅显示聚合信息。配置与数据路径对应生成本 Notebook 的那次运行。

In [2]:
assert len(raw) == len(cleaned)
assert raw.customerID.str.strip().tolist() == cleaned.customerID.tolist()
assert raw.Churn.str.strip().tolist() == cleaned.Churn.tolist()
assert cleaned.churn_label.sum() == raw.Churn.str.strip().eq('Yes').sum()
assert cleaned.TotalCharges.isna().sum() == raw.TotalCharges.str.strip().eq('').sum()
assert cleaned.internet_applicable.sum() == raw.InternetService.str.strip().ne('No').sum()
pd.testing.assert_frame_equal(cleaned, cleaner.transform(cleaned))
cleaned[['TotalCharges', 'tenure']].isna().sum().to_frame('缺失数')

,缺失数
TotalCharges,11
tenure,0


## 训练内拟合与编码
固定种子分层切分仅演示接口。验证集不参与中位数、均值或标准差拟合；这不是 W5 模型评估。

In [3]:
X, y, ids = split_xy(cleaned, config)
train, valid = train_test_split(np.arange(len(X)), test_size=config['test_size'], random_state=config['seed'], stratify=y)
pipeline = build_preprocessor(config)
X_train = pipeline.fit_transform(X.iloc[train])
X_valid = pipeline.transform(X.iloc[valid])
assert np.isfinite(X_train).all() and np.isfinite(X_valid).all()
assert not set(train) & set(valid)
print('训练矩阵:', X_train.shape, '验证矩阵:', X_valid.shape)
print('X 输入不含ID和标签:', not {'customerID', 'Churn', 'churn_label'} & set(X.columns))

训练矩阵: (5634, 49) 验证矩阵: (1409, 49)
X 输入不含ID和标签: True


In [4]:
num = pipeline.named_steps['columns'].named_transformers_['numeric']
assert np.allclose(num.named_steps['imputer'].statistics_, X.iloc[train][config['features']['numeric']].median().fillna(0))
pd.DataFrame({'字段': config['features']['numeric'], '训练中位数': num.named_steps['imputer'].statistics_, '训练缩放均值': num.named_steps['scaler'].mean_})

,字段,训练中位数,训练缩放均值
0,tenure,29.000,32.485091
1,MonthlyCharges,70.500,64.929961
2,TotalCharges,1398.125,2301.319950


## 累计费用差异诊断
历史价格、优惠和账单日期不可见。累计费用与当前月费乘以年限的差异只用于检查，不被当作修复目标。

In [5]:
difference = cleaned.TotalCharges - cleaned.tenure * cleaned.MonthlyCharges
difference.describe(percentiles=[.01, .25, .5, .75, .99]).to_frame('累计费用差异')

,累计费用差异
count,7032.000000
mean,0.153193
std,67.255326
min,-370.850000
1%,-190.138000
25%,-28.650000
50%,0.000000
75%,28.700000
99%,191.460500
max,373.250000


## 交付与限制
清洗规则见 `reports/w3_cleaning_rules.md`；12 项真实时间特征设计见 `reports/feature_engineering_design_v1.md`。当前没有事件时间数据，实际时序计算为零；49列仅是 W3 接口示例，不代替 W4 的最终特征设计。详细机器核验见 `reports/w3_verification.json`。